In [1]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

In [2]:
# Vocabulary words used for recognition
words = ["yes", "no", "up", "down"]

# Dataset directory paths
train_path = "dataset/train"
test_path = "dataset/test"


# Function to extract MFCC features from audio file
def extract_mfcc(file_path, n_mfcc=13):
    signal, sr = librosa.load(file_path, sr=None)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=n_mfcc)
    return mfcc.T


In [3]:
# Function to compute DTW distance between two MFCC sequences
def dtw_distance(seq1, seq2):
    n, m = len(seq1), len(seq2)

    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = np.linalg.norm(seq1[i - 1] - seq2[j - 1])

            dtw[i, j] = cost + min(
                dtw[i - 1, j],
                dtw[i, j - 1],
                dtw[i - 1, j - 1]
            )

    return dtw[n, m]


# Store training MFCC features and labels
train_features = []
train_labels = []

for word in words:
    folder = os.path.join(train_path, word)

    for file in os.listdir(folder):
        mfcc = extract_mfcc(os.path.join(folder, file))

        train_features.append(mfcc)
        train_labels.append(word)


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset/train\\yes'

In [ ]:
# Testing and prediction
true_labels = []
pred_labels = []

for word in words:
    folder = os.path.join(test_path, word)

    for file in os.listdir(folder):
        test_mfcc = extract_mfcc(os.path.join(folder, file))

        distances = []

        # Compare with every training sample
        for train_mfcc in train_features:
            dist = dtw_distance(test_mfcc, train_mfcc)
            distances.append(dist)

        # Predict label of nearest training template
        pred_word = train_labels[np.argmin(distances)]

        true_labels.append(word)
        pred_labels.append(pred_word)


# Compute recognition accuracy
accuracy = accuracy_score(true_labels, pred_labels)

# Generate confusion matrix
cm = confusion_matrix(true_labels, pred_labels, labels=words)

print("Recognition Accuracy:", round(accuracy * 100, 2), "%")


In [ ]:
# Plot confusion matrix
plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=words,
    yticklabels=words,
    cmap='Blues'
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.tight_layout()
plt.show()